# Netflix Recommendation System - Memory-Optimized Kaggle Notebook

> **Dataset**: [Netflix Prize Data](https://www.kaggle.com/datasets/netflix-inc/netflix-prize-data)  
> **Goal**: Load ~100M ratings with strict memory management, train KNN/SVD/NeuMF on a sample.

## Memory Strategy

| Technique | Benefit |
|-----------|----------|
| `int8` ratings + `int32` IDs at parse time | ~60 % vs default `int64` |
| Chunked per-file sampling | Peak RAM = one file at a time |
| Skip date parse during load; parse on sample | ~200 MB saved per file |
| `del` + `gc.collect()` between stages | Frees unreachable objects immediately |
| Parquet caching | No re-parse on notebook re-run |
| Validation in 4096-row mini-batches | Avoids GPU OOM |
| `torch.cuda.empty_cache()` each epoch | Returns allocator memory to CUDA pool |


## 0  Environment Setup & Memory Utilities

In [ ]:
import os, gc, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from contextlib import contextmanager

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

# ---- memory helpers --------------------------------------------------------

def mem_mb(obj=None):
    """RSS of current process (MB), or DataFrame size."""
    if isinstance(obj, pd.DataFrame):
        return obj.memory_usage(deep=True).sum() / 1024**2
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / 1024**2
    except ImportError:
        return -1.0


def reduce_mem_df(df, verbose=True):
    """Downcast integer/float columns to the smallest safe dtype."""
    before = mem_mb(df)
    for col in df.select_dtypes(include=['integer','float']).columns:
        c_min, c_max = df[col].min(), df[col].max()
        if pd.api.types.is_integer_dtype(df[col]):
            for dtype in [np.int8, np.int16, np.int32]:
                ii = np.iinfo(dtype)
                if c_min >= ii.min and c_max <= ii.max:
                    df[col] = df[col].astype(dtype); break
        else:
            fi = np.finfo(np.float32)
            if c_min >= fi.min and c_max <= fi.max:
                df[col] = df[col].astype(np.float32)
    after = mem_mb(df)
    if verbose:
        print(f'  reduce_mem_df: {before:.1f} -> {after:.1f} MB  '
              f'({(before-after)/before*100:.1f}% saved)')
    return df


@contextmanager
def mem_snapshot(label='block'):
    """Print RAM delta and wall-time for a code block."""
    before = mem_mb()
    t0 = time.time()
    yield
    after = mem_mb()
    print(f'[{label}] RAM: {before:.0f}->{after:.0f} MB  '
          f'(delta {after-before:+.0f} MB)  time: {time.time()-t0:.1f}s')


print(f'Python {sys.version}')
print(f'Pandas {pd.__version__}  |  NumPy {np.__version__}')
print(f'Initial RAM: {mem_mb():.0f} MB')


## 1  Configuration

In [ ]:
# ---- Kaggle dataset path detection ----------------------------------------
_KAGGLE_PATHS = [
    '/kaggle/input/netflix-prize-data',
    '/kaggle/input/netflix-inc-netflix-prize-data',
    '/kaggle/input/netflix-prize-data-1',
]
RAW_DATA_DIR = next((p for p in _KAGGLE_PATHS if os.path.isdir(p)), 'data/raw')

PROCESSED_DIR = '/kaggle/working/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ---- Hyper-parameters ------------------------------------------------------
RANDOM_SEED         = 42
TEST_SIZE           = 0.20
SAMPLE_FRACTION     = 0.10   # 10% -> ~10 M rows, safe on Kaggle 16 GB
MIN_USER_RATINGS    = 20
MIN_MOVIE_RATINGS   = 50
RELEVANCE_THRESHOLD = 3.5
TOP_K               = 10

KNN_USER_PARAMS = {'k': 40, 'sim_options': {'name': 'cosine',  'user_based': True}}
KNN_ITEM_PARAMS = {'k': 40, 'sim_options': {'name': 'pearson', 'user_based': False}}
SVD_PARAMS      = {'n_factors': 100, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}
NCF_PARAMS      = {
    'embedding_dim': 64, 'hidden_layers': [128, 64, 32],
    'dropout': 0.2, 'lr': 0.001, 'batch_size': 2048,
    'epochs': 10, 'patience': 3,
}

np.random.seed(RANDOM_SEED)
print('Config OK')
print(f'Raw data dir : {RAW_DATA_DIR}')
print(f'Processed dir: {PROCESSED_DIR}')
print(f'Sample frac  : {SAMPLE_FRACTION}')


## 2  Memory-Optimised Data Loading

**Key improvements over the original `data_loader.py`**:
1. `int8` for ratings, `int32` for user IDs **at parse time** — not after.
2. Dates stored as raw strings during the heavy parse; only converted on the small sample.
3. One file at a time: parse → sample → free the big frame → next file.
4. Parquet cache: rerun skips all parsing.


In [ ]:
PARQUET_PATH = os.path.join(PROCESSED_DIR, 'sampled_ratings.parquet')


def parse_combined_lean(filepath):
    """
    Memory-lean parser for Netflix combined_data_N.txt.
    Uses small dtypes from the first insert, not a post-hoc cast.
    """
    fsize = os.path.getsize(filepath) / 1024**2
    print(f'Parsing {os.path.basename(filepath)} ({fsize:.0f} MB)...')
    user_ids, movie_ids, ratings, dates = [], [], [], []
    current_movie = None
    with open(filepath, 'r') as f:
        for line in tqdm(f, desc=os.path.basename(filepath), leave=False):
            line = line.rstrip('\n')
            if not line:
                continue
            if line.endswith(':'):
                current_movie = int(line[:-1])
            else:
                uid, rat, dt = line.split(',')
                user_ids.append(int(uid))
                movie_ids.append(current_movie)
                ratings.append(int(rat))
                dates.append(dt)
    df = pd.DataFrame({
        'user_id':  pd.array(user_ids,  dtype='int32'),
        'movie_id': pd.array(movie_ids, dtype='int16'),
        'rating':   pd.array(ratings,   dtype='int8'),
        'date':     dates,
    })
    del user_ids, movie_ids, ratings, dates
    gc.collect()
    print(f'  -> {len(df):,} rows  |  {mem_mb(df):.0f} MB')
    return df


def load_and_sample_all(data_dir, frac=SAMPLE_FRACTION, seed=RANDOM_SEED):
    """
    Load all 4 combined_data files ONE AT A TIME.
    Peak RAM = one full file + one sample (not all 4 at once).
    """
    parts = []
    for i in range(1, 5):
        fp = os.path.join(data_dir, f'combined_data_{i}.txt')
        if not os.path.exists(fp):
            print(f'Not found: {fp} -- skipping')
            continue
        with mem_snapshot(f'file {i}'):
            df_full = parse_combined_lean(fp)
            uids    = df_full['user_id'].unique()
            rng     = np.random.default_rng(seed + i)
            sel     = rng.choice(uids, size=max(1, int(len(uids)*frac)), replace=False)
            sample  = df_full[df_full['user_id'].isin(sel)].copy()
            del df_full; gc.collect()
            print(f'  Sampled: {len(sample):,} rows  |  {mem_mb(sample):.0f} MB')
            parts.append(sample)
    combined = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    print(f'Combined sample: {combined.shape}  |  {mem_mb(combined):.0f} MB')
    return combined


# ---- load or restore from cache --------------------------------------------
if os.path.exists(PARQUET_PATH):
    print('Loading cached parquet...')
    with mem_snapshot('parquet load'):
        ratings_df = pd.read_parquet(PARQUET_PATH)
    print(f'Loaded: {ratings_df.shape}  |  {mem_mb(ratings_df):.0f} MB')
else:
    with mem_snapshot('full load + sample'):
        ratings_df = load_and_sample_all(RAW_DATA_DIR)
    ratings_df.to_parquet(PARQUET_PATH, index=False)
    print(f'Saved cache: {PARQUET_PATH}')

print(f'RAM after load: {mem_mb():.0f} MB')


## 3  Filtering, Encoding & Train/Test Split

In [ ]:
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix

print(f'Before filtering: {len(ratings_df):,}')

uc = ratings_df['user_id'].value_counts()
ratings_df = ratings_df[ratings_df['user_id'].isin(uc[uc >= MIN_USER_RATINGS].index)]
del uc; gc.collect()
print(f'After user filter  (>={MIN_USER_RATINGS}): {len(ratings_df):,}')

mc = ratings_df['movie_id'].value_counts()
ratings_df = ratings_df[ratings_df['movie_id'].isin(mc[mc >= MIN_MOVIE_RATINGS].index)]
del mc; gc.collect()
print(f'After movie filter (>={MIN_MOVIE_RATINGS}): {len(ratings_df):,}')

# parse dates on the smaller sample only
with mem_snapshot('date parse'):
    ratings_df['date'] = pd.to_datetime(ratings_df['date'], format='%Y-%m-%d')

user_enc = LabelEncoder()
item_enc = LabelEncoder()
ratings_df['user_idx'] = user_enc.fit_transform(ratings_df['user_id']).astype(np.int32)
ratings_df['item_idx'] = item_enc.fit_transform(ratings_df['movie_id']).astype(np.int32)

N_USERS  = len(user_enc.classes_)
N_ITEMS  = len(item_enc.classes_)
sparsity = 1.0 - len(ratings_df) / (N_USERS * N_ITEMS)
print(f'Users: {N_USERS:,}  Movies: {N_ITEMS:,}  Ratings: {len(ratings_df):,}')
print(f'Sparsity: {sparsity*100:.4f}%  |  RAM: {mem_mb():.0f} MB')


In [ ]:
# Memory-efficient temporal split
# Avoids the original double full-size rank/count columns from preprocessing.py
with mem_snapshot('temporal split'):
    dfs = ratings_df.sort_values(['user_id', 'date'])
    dfs['cc']  = dfs.groupby('user_id').cumcount()
    tot        = dfs.groupby('user_id')['user_id'].transform('count')
    cutoff     = (tot * (1 - TEST_SIZE)).astype(int)
    is_test    = dfs['cc'] >= cutoff
    train_df   = dfs[~is_test].drop(columns=['cc']).reset_index(drop=True)
    test_df    = dfs[ is_test].drop(columns=['cc']).reset_index(drop=True)
    del dfs, tot, cutoff, is_test; gc.collect()

print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')
overlap = len(set(train_df['user_id']) & set(test_df['user_id']))
print(f'Users in both sets: {overlap:,}  |  RAM: {mem_mb():.0f} MB')


## 4  Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ratings_df['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Count')

u_cnt = ratings_df.groupby('user_id').size()
axes[1].hist(u_cnt, bins=60, color='coral', log=True, edgecolor='white')
axes[1].set_title('Ratings per User (log)'); axes[1].set_xlabel('# Ratings')

m_cnt = ratings_df.groupby('movie_id').size()
axes[2].hist(m_cnt, bins=60, color='mediumseagreen', log=True, edgecolor='white')
axes[2].set_title('Ratings per Movie (log)'); axes[2].set_xlabel('# Ratings')

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'eda_distributions.png'), dpi=100)
plt.show()
del u_cnt, m_cnt; gc.collect()

mem_col = ratings_df.memory_usage(deep=True).drop('Index') / 1024**2
print('Memory by column (MB):')
print(mem_col.sort_values(ascending=False).to_string())
print(f'Total: {mem_col.sum():.1f} MB')


## 5  KNN Collaborative Filtering

In [ ]:
from surprise import Reader, Dataset, KNNWithMeans

reader = Reader(rating_scale=(1, 5))
with mem_snapshot('surprise dataset'):
    s_data     = Dataset.load_from_df(train_df[['user_id','movie_id','rating']], reader)
    s_trainset = s_data.build_full_trainset()
print(f'Surprise: {s_trainset.n_users} users, {s_trainset.n_items} items  |  RAM: {mem_mb():.0f} MB')

with mem_snapshot('KNN-User fit'):
    knn_user = KNNWithMeans(**KNN_USER_PARAMS)
    knn_user.fit(s_trainset)
print('KNN-User trained')

with mem_snapshot('KNN-Item fit'):
    knn_item = KNNWithMeans(**KNN_ITEM_PARAMS)
    knn_item.fit(s_trainset)
print('KNN-Item trained')


## 6  SVD

In [ ]:
from surprise import SVD

with mem_snapshot('SVD fit'):
    svd_model = SVD(**SVD_PARAMS, random_state=RANDOM_SEED)
    svd_model.fit(s_trainset)
print(f'SVD trained  |  RAM: {mem_mb():.0f} MB')


## 7  Neural Collaborative Filtering (NeuMF)

Memory optimisations vs original `neural_cf.py`:
- `batch_size=2048` reduces DataLoader overhead per epoch.
- `pin_memory=True` only when CUDA is present.
- `torch.cuda.empty_cache()` after every epoch returns allocator memory to CUDA pool.
- Validation done in 4096-row mini-batches — prevents GPU OOM.


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset as TDS, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')


class RatingsDataset(TDS):
    def __init__(self, df):
        # numpy arrays avoid keeping the DataFrame alive in DataLoader workers
        self.u = df['user_idx'].values.astype(np.int32)
        self.i = df['item_idx'].values.astype(np.int32)
        self.r = df['rating'].values.astype(np.float32)
    def __len__(self): return len(self.r)
    def __getitem__(self, idx):
        return (torch.tensor(self.u[idx], dtype=torch.long),
                torch.tensor(self.i[idx], dtype=torch.long),
                torch.tensor(self.r[idx], dtype=torch.float32))


class NeuMF(nn.Module):
    def __init__(self, n_u, n_i, d=64, h=[128,64,32], drop=0.2):
        super().__init__()
        self.ug = nn.Embedding(n_u, d); self.ig = nn.Embedding(n_i, d)
        self.um = nn.Embedding(n_u, d); self.im = nn.Embedding(n_i, d)
        layers, dim = [], d*2
        for hd in h:
            layers += [nn.Linear(dim, hd), nn.ReLU(), nn.Dropout(drop)]; dim=hd
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(d+h[-1], 1)
    def forward(self, u, i):
        gmf = self.ug(u) * self.ig(i)
        mlp = self.mlp(torch.cat([self.um(u), self.im(i)], -1))
        return self.out(torch.cat([gmf, mlp], -1)).squeeze(-1)


ncf = NeuMF(N_USERS, N_ITEMS,
            d=NCF_PARAMS['embedding_dim'],
            h=NCF_PARAMS['hidden_layers'],
            drop=NCF_PARAMS['dropout']).to(DEVICE)
n_p = sum(p.numel() for p in ncf.parameters() if p.requires_grad)
print(f'NeuMF parameters: {n_p:,}  |  RAM: {mem_mb():.0f} MB')


In [ ]:
PIN          = DEVICE.type == 'cuda'
train_ds     = RatingsDataset(train_df)
val_ds       = RatingsDataset(test_df)
train_loader = DataLoader(train_ds, batch_size=NCF_PARAMS['batch_size'],
                          shuffle=True, pin_memory=PIN, num_workers=2 if PIN else 0)
criterion    = nn.MSELoss()
optimizer    = torch.optim.Adam(ncf.parameters(), lr=NCF_PARAMS['lr'])
SAVE_PATH    = os.path.join(PROCESSED_DIR, 'best_ncf.pt')

best_val, pat = float('inf'), 0
t_losses, v_rmses = [], []

for epoch in range(NCF_PARAMS['epochs']):
    # -- train --
    ncf.train(); total = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NCF_PARAMS["epochs"]}', leave=False)
    for u, i, r in pbar:
        u,i,r = u.to(DEVICE), i.to(DEVICE), r.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(ncf(u,i), r)
        loss.backward(); optimizer.step()
        total += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    avg = total / len(train_loader); t_losses.append(avg)

    # -- validate in mini-batches (avoids GPU OOM) --
    ncf.eval(); preds, tgts = [], []
    with torch.no_grad():
        for vu,vi,vr in DataLoader(val_ds, batch_size=4096):
            preds.extend(ncf(vu.to(DEVICE), vi.to(DEVICE)).cpu().numpy())
            tgts.extend(vr.numpy())
    rmse = float(np.sqrt(np.mean((np.array(preds)-np.array(tgts))**2)))
    v_rmses.append(rmse)

    print(f'Epoch {epoch+1:02d}  loss={avg:.4f}  val_rmse={rmse:.4f}  RAM={mem_mb():.0f}MB')

    if DEVICE.type == 'cuda': torch.cuda.empty_cache()  # free allocator cache

    if rmse < best_val:
        best_val, pat = rmse, 0
        torch.save(ncf.state_dict(), SAVE_PATH)
    else:
        pat += 1
        if pat >= NCF_PARAMS['patience']:
            print(f'Early stopping at epoch {epoch+1}'); break

ncf.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
print(f'Best Val RMSE: {best_val:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t_losses, 'o-', color='steelblue')
axes[0].set_title('NCF Training Loss'); axes[0].set_xlabel('Epoch')
axes[1].plot(v_rmses,  's-', color='coral')
axes[1].set_title('NCF Val RMSE'); axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'ncf_curves.png'), dpi=100)
plt.show()


## 8  Evaluation

In [ ]:
from surprise import accuracy as sa

def eval_surprise(model, df, name, max_r=50_000):
    df_ev = df.sample(min(max_r, len(df)), random_state=RANDOM_SEED)
    preds = [model.predict(str(r.user_id), str(r.movie_id), r_ui=float(r.rating))
             for r in df_ev[['user_id','movie_id','rating']].itertuples(index=False)]
    rmse = sa.rmse(preds, verbose=False)
    mae  = sa.mae( preds, verbose=False)
    print(f'{name:<20} RMSE={rmse:.4f}  MAE={mae:.4f}')
    return rmse, mae

def eval_ncf(model, df, device=DEVICE):
    model.eval(); ds = RatingsDataset(df)
    p, t = [], []
    with torch.no_grad():
        for u,i,r in DataLoader(ds, batch_size=4096):
            p.extend(model(u.to(device), i.to(device)).cpu().numpy())
            t.extend(r.numpy())
    p = np.clip(np.array(p), 1, 5); t = np.array(t)
    rmse = float(np.sqrt(np.mean((p-t)**2)))
    mae  = float(np.mean(np.abs(p-t)))
    print(f'{"NeuMF-NCF":<20} RMSE={rmse:.4f}  MAE={mae:.4f}')
    return rmse, mae

print('='*48)
ku_r, ku_m = eval_surprise(knn_user,  test_df, 'KNN-User')
ki_r, ki_m = eval_surprise(knn_item,  test_df, 'KNN-Item')
sv_r, sv_m = eval_surprise(svd_model, test_df, 'SVD')
nc_r, nc_m = eval_ncf(ncf, test_df)
print(f'RAM: {mem_mb():.0f} MB')


In [ ]:
results = pd.DataFrame([
    {'Model':'KNN-User',  'RMSE':ku_r, 'MAE':ku_m},
    {'Model':'KNN-Item',  'RMSE':ki_r, 'MAE':ki_m},
    {'Model':'SVD',       'RMSE':sv_r, 'MAE':sv_m},
    {'Model':'NeuMF-NCF', 'RMSE':nc_r, 'MAE':nc_m},
]).set_index('Model').round(4)
print(results.to_string())

clrs = ['steelblue','coral','mediumseagreen','mediumpurple']
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
results['RMSE'].plot(kind='bar', ax=ax[0], color=clrs, edgecolor='white')
ax[0].set_title('RMSE (lower=better)'); ax[0].tick_params(axis='x', rotation=20)
results['MAE'].plot(kind='bar',  ax=ax[1], color=clrs, edgecolor='white')
ax[1].set_title('MAE (lower=better)');  ax[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'model_comparison.png'), dpi=100)
plt.show()


## 9  Sample Recommendations

In [ ]:
# load movie titles
TITLES_PATH = None
for tp in [os.path.join(RAW_DATA_DIR,'movie_titles.csv'), os.path.join(RAW_DATA_DIR,'movie_titles.txt')]:
    if os.path.exists(tp): TITLES_PATH = tp; break

titles_df = None
if TITLES_PATH:
    recs = []
    with open(TITLES_PATH,'r',encoding='ISO-8859-1') as f:
        for line in f:
            parts = line.strip().split(',',2)
            if len(parts)==3: recs.append({'movie_id':int(parts[0]),'year':parts[1],'title':parts[2].strip()})
    titles_df = pd.DataFrame(recs)
    print(f'Loaded {len(titles_df)} movie titles')
else:
    print('movie_titles not found')

def get_title(mid):
    if titles_df is None: return str(mid)
    row = titles_df[titles_df['movie_id']==mid]
    return row['title'].values[0] if len(row) else str(mid)

rng         = np.random.default_rng(RANDOM_SEED)
sample_user = int(rng.choice(test_df['user_id'].unique()))
rated       = set(ratings_df[ratings_df['user_id']==sample_user]['movie_id'])
all_movies  = list(ratings_df['movie_id'].unique())
print(f'User {sample_user}  |  rated {len(rated)} movies')

# SVD top-N
svd_recs = sorted(
    [(m, svd_model.predict(str(sample_user),str(m)).est) for m in all_movies if m not in rated],
    key=lambda x:x[1], reverse=True)
print(f'\nTop-{TOP_K} SVD recs for user {sample_user}:')
for rank,(mid,sc) in enumerate(svd_recs[:TOP_K],1):
    print(f'  {rank:2d}. {get_title(mid):<45} (est={sc:.2f})')


In [ ]:
# NeuMF top-N (batched)
cands = [m for m in all_movies if m not in rated]
try:
    u_idx  = int(user_enc.transform([sample_user])[0])
    ci, cm = [], []
    for m in cands:
        try: ci.append(int(item_enc.transform([m])[0])); cm.append(m)
        except ValueError: pass
    ncf.eval(); scores = []
    BS = 512
    with torch.no_grad():
        for s in range(0,len(ci),BS):
            i_t = torch.tensor(ci[s:s+BS],dtype=torch.long).to(DEVICE)
            u_t = torch.tensor([u_idx]*len(i_t),dtype=torch.long).to(DEVICE)
            scores.extend(ncf(u_t,i_t).cpu().numpy())
    ncf_recs = sorted(zip(cm,scores),key=lambda x:x[1],reverse=True)
    print(f'\nTop-{TOP_K} NeuMF recs for user {sample_user}:')
    for rank,(mid,sc) in enumerate(ncf_recs[:TOP_K],1):
        print(f'  {rank:2d}. {get_title(mid):<45} (est={sc:.2f})')
except Exception as e:
    print(f'NeuMF recs error: {e}')


## 10  Full Memory Diagnostic Report

In [ ]:
import sys
try:
    import psutil
    mi = psutil.Process(os.getpid()).memory_info()
    vm = psutil.virtual_memory()
    print('='*50, '\n  MEMORY DIAGNOSTIC\n' + '='*50)
    print(f'  Process RSS      : {mi.rss/1024**3:.2f} GB')
    print(f'  Process VMS      : {mi.vms/1024**3:.2f} GB')
    print(f'  System total RAM : {vm.total/1024**3:.2f} GB')
    print(f'  System available : {vm.available/1024**3:.2f} GB')
    print(f'  System used %    : {vm.percent:.1f}%')
except ImportError:
    print(f'psutil not available. Est RAM: {mem_mb():.0f} MB')

if DEVICE.type == 'cuda':
    print(f'  GPU allocated  : {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')
    print(f'  GPU reserved   : {torch.cuda.memory_reserved(0)/1024**3:.2f} GB')

print('\nTop 15 globals by size (MB):')
objs = {k: sys.getsizeof(v)/1024**2 for k,v in globals().items()
        if not k.startswith('_') and sys.getsizeof(v) > 1}
for nm, sz in sorted(objs.items(), key=lambda x:x[1], reverse=True)[:15]:
    print(f'  {nm:<28} {sz:>8.2f} MB')


## 11  Save Outputs

In [ ]:
import pickle

enc_path = os.path.join(PROCESSED_DIR,'encoders.pkl')
with open(enc_path,'wb') as f:
    pickle.dump({'user_enc':user_enc,'item_enc':item_enc}, f)
print(f'Encoders -> {enc_path}')

results.to_csv(os.path.join(PROCESSED_DIR,'results.csv'))
print(f'Results  -> {PROCESSED_DIR}/results.csv')
print(f'NCF      -> {SAVE_PATH}')

print(f'\nFinal RAM: {mem_mb():.0f} MB')
print('Notebook complete! Outputs:')
for fn in os.listdir(PROCESSED_DIR):
    sz = os.path.getsize(os.path.join(PROCESSED_DIR,fn))/1024**2
    print(f'  {fn:<40} {sz:.1f} MB')
